## Step 1 – Hello, Data!

In this step, I load the raw e-commerce transaction data using pandas.
The dataset contains 500 transaction records with information about
customers, products, prices, quantities, coupon codes and shipping cities.
The first three rows are displayed to verify that the data is loaded correctly.

In [12]:
import pandas as pd
import numpy as np
import json

sales = pd.read_csv("../data/ecommerce_500.csv")
metadata = pd.read_csv("../data/metadata.csv")

print("Sales dataset shape:", sales.shape)
print("Metadata dataset shape:", metadata.shape)

sales.head(3)

Sales dataset shape: (500, 7)
Metadata dataset shape: (15, 8)


,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,01-01-2025,CUST1078,Tablet,583.56,3.0,SAVE15,Oakville
1,01-01-2025,CUST1055,Smartphone,1103.70,4.0,WELCOME10,Brampton
2,02-01-2025,CUST1056,Tablet,378.12,5.0,WELCOME10,Toronto


## Step 2 – Pick the Right Container

A dictionary is suitable for representing an individual transaction because
each value can be accessed using a meaningful key such as customer_id, product,
price, and quantity. 

A set is useful for finding unique shipping cities because duplicate values are automatically removed. 

A namedtuple could represent a fixed transaction structure, but a dictionary is more flexible when additional fields need to be added.

In [13]:
transaction = {
    "customer_id": "CUST1078",
    "product": "Tablet",
    "price": 583.56,
    "quantity": 3,
    "shipping_city": "Oakville"
}

print(transaction)

{'customer_id': 'CUST1078', 'product': 'Tablet', 'price': 583.56, 'quantity': 3, 'shipping_city': 'Oakville'}


In [14]:
class Transaction:
    
    def __init__(self, customer_id, product, price, quantity, coupon_code):
        self.customer_id = customer_id
        self.product = product
        self.price = price
        self.quantity = quantity
        self.coupon_code = coupon_code
    
    def total(self):
        return self.price * self.quantity
    
    def clean(self):
        self.product = self.product.strip().title()
        self.coupon_code = self.coupon_code.strip().upper()
        return self

In [15]:
row = sales.iloc[0]

transaction = Transaction(
    row["customer_id"],
    row["product"],
    row["price"],
    row["quantity"],
    row["coupon_code"]
)

transaction.clean()

print("Customer:", transaction.customer_id)
print("Product:", transaction.product)
print("Coupon:", transaction.coupon_code)
print("Transaction total:", transaction.total())

Customer: CUST1078
Product: Tablet
Coupon: SAVE15
Transaction total: 1750.6799999999998


## Step 4 – Bulk Loaded

The transaction DataFrame is converted into a list of dictionaries.
This provides another way to represent each record using key-value pairs.
The same approach is used with the secondary city metadata dataset.

In [16]:
transactions = sales.to_dict(orient="records")

print("Number of transaction dictionaries:", len(transactions))

transactions[:2]

Number of transaction dictionaries: 500


[{'date': '01-01-2025',
  'customer_id': 'CUST1078',
  'product': 'Tablet',
  'price': 583.56,
  'quantity': 3.0,
  'coupon_code': 'SAVE15',
  'shipping_city': 'Oakville'},
 {'date': '01-01-2025',
  'customer_id': 'CUST1055',
  'product': 'Smartphone',
  'price': 1103.7,
  'quantity': 4.0,
  'coupon_code': 'WELCOME10',
  'shipping_city': 'Brampton'}]

In [17]:
city_metadata = metadata.to_dict(orient="records")

city_metadata[:2]

[{'city': 'Toronto',
  'province': 'Ontario',
  'country': 'Canada',
  'province_code': 'ON',
  'timezone': 'America/Toronto',
  'population': 2794356,
  'latitude': 43.6532,
  'longitude': -79.3832},
 {'city': 'Mississauga',
  'province': 'Ontario',
  'country': 'Canada',
  'province_code': 'ON',
  'timezone': 'America/Toronto',
  'population': 717961,
  'latitude': 43.589,
  'longitude': -79.6441}]

## Step 5 – Quick Profiling

I use basic statistics to understand the price values and check the number of unique shipping cities. This gives an initial overview of the dataset before cleaning.

In [18]:
print("Minimum price:", sales["price"].min())
print("Mean price:", sales["price"].mean())
print("Maximum price:", sales["price"].max())

unique_cities = set(sales["shipping_city"].dropna())

print("Number of unique cities:", len(unique_cities))
print("Unique cities:", unique_cities)


Minimum price: 10.17
Mean price: 381.87757999999997
Maximum price: 1799.9
Number of unique cities: 15
Unique cities: {'Oakville', 'Toronto', 'Burlington', 'Ottawa', 'Hamilton', 'Brampton', 'Waterloo', 'Mississauga', 'London', 'Cambridge', 'Vaughan', 'Windsor', 'Markham', 'Richmond Hill', 'Kitchener'}


## Step 6 – Spot the Grime

I check the dataset for common data-quality problems such as missing values, duplicate records, incorrect data types and invalid price or quantity values.

In [19]:
print("Missing values:")
print(sales.isnull().sum())

print("\nDuplicate rows:", sales.duplicated().sum())

print("\nData types:")
print(sales.dtypes)

print("\nNegative prices:", (sales["price"] < 0).sum())

print("Invalid quantities:", (sales["quantity"] <= 0).sum())

Missing values:
date             0
customer_id      0
product          0
price            0
quantity         1
coupon_code      0
shipping_city    1
dtype: int64

Duplicate rows: 0

Data types:
date                 str
customer_id          str
product              str
price            float64
quantity         float64
coupon_code          str
shipping_city        str
dtype: object

Negative prices: 0
Invalid quantities: 0


## Step 7 – Cleaning Rules

I clean the transaction data by removing duplicates, standardizing text fields, converting data types, and removing invalid records. The cleaning rules are implemented inside the `clean()` method.

In [20]:
class DataCleaner:
    
    def __init__(self, data):
        self.df = data.copy()
    
    def clean(self):
        
        # Remove duplicate records
        self.df = self.df.drop_duplicates()
        
        # Clean text fields
        self.df["shipping_city"] = (
            self.df["shipping_city"]
            .astype(str)
            .str.strip()
            .str.title()
        )
        
        self.df["product"] = (
            self.df["product"]
            .astype(str)
            .str.strip()
        )
        
        self.df["coupon_code"] = (
            self.df["coupon_code"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
        
        # Convert numeric columns
        self.df["price"] = pd.to_numeric(
            self.df["price"], errors="coerce"
        )
        
        self.df["quantity"] = pd.to_numeric(
            self.df["quantity"], errors="coerce"
        )
        
        # Convert date
        self.df["date"] = pd.to_datetime(
            self.df["date"], errors="coerce"
        )
        
        # Remove missing required values
        self.df = self.df.dropna(
            subset=[
                "date",
                "customer_id",
                "product",
                "price",
                "quantity",
                "shipping_city"
            ]
        )
        
        # Remove invalid numeric values
        self.df = self.df[
            (self.df["price"] >= 0) &
            (self.df["quantity"] > 0)
        ]
        
        return self.df

In [21]:
before_count = len(sales)

cleaner = DataCleaner(sales)
clean_df = cleaner.clean()

after_count = len(clean_df)

print("Rows before cleaning:", before_count)
print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

Rows before cleaning: 500
Rows after cleaning: 202
Rows removed: 298


## Step 8 – Transformations

I transform the coupon codes into numeric discount percentages. I then use the discount to calculate the discounted price and transaction revenue.

In [23]:
clean_df["discount_percent"] = (
    clean_df["coupon_code"]
    .str.extract(r"(\d+)")[0]
    .fillna(0)
    .astype(float)
)

clean_df[
    ["coupon_code", "discount_percent"]
].head(10)

clean_df["discounted_price"] = (
    clean_df["price"] *
    (1 - clean_df["discount_percent"] / 100)
)

clean_df["revenue"] = (
    clean_df["discounted_price"] *
    clean_df["quantity"]
)

clean_df[
    [
        "price",
        "quantity",
        "coupon_code",
        "discount_percent",
        "discounted_price",
        "revenue"
    ]
].head()

,price,quantity,coupon_code,discount_percent,discounted_price,revenue
0,583.56,3.0,SAVE15,15.0,496.026,1488.078
1,1103.70,4.0,WELCOME10,10.0,993.330,3973.320
2,378.12,5.0,WELCOME10,10.0,340.308,1701.540
3,29.47,4.0,TECH20,20.0,23.576,94.304
5,214.30,4.0,SAVE15,15.0,182.155,728.620


## Step 9 – Feature Engineering

I create new features from the cleaned data. The `days_since_purchase` feature measures how many days have passed since each purchase relative to the latest purchase date.

In [24]:
reference_date = clean_df["date"].max()

clean_df["days_since_purchase"] = (
    reference_date - clean_df["date"]
).dt.days

clean_df["gross_value"] = (
    clean_df["price"] *
    clean_df["quantity"]
)

clean_df[
    [
        "date",
        "price",
        "quantity",
        "days_since_purchase",
        "gross_value"
    ]
].head()

,date,price,quantity,days_since_purchase,gross_value
0,2025-01-01,583.56,3.0,345,1750.68
1,2025-01-01,1103.70,4.0,345,4414.80
2,2025-02-01,378.12,5.0,314,1890.60
3,2025-04-01,29.47,4.0,255,117.88
5,2025-05-01,214.30,4.0,225,857.20


## Step 10 – Mini-Aggregation

I merge the transaction data with the secondary city metadata using the shipping city. I then calculate total revenue for each city.

In [25]:
enriched_df = clean_df.merge(
    metadata,
    left_on="shipping_city",
    right_on="city",
    how="left"
)

enriched_df.head()

print(
    "Unmatched cities:",
    enriched_df["city"].isna().sum()
)

revenue_by_city = (
    enriched_df
    .groupby("shipping_city")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

revenue_by_city

city_summary = (
    enriched_df
    .groupby("shipping_city")
    .agg(
        total_revenue=("revenue", "sum"),
        total_quantity=("quantity", "sum"),
        customers=("customer_id", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

city_summary


Unmatched cities: 0


,total_revenue,total_quantity,customers
shipping_city,,,
Toronto,24990.6680,43.0,11
Cambridge,19931.9310,63.0,22
Waterloo,18461.8585,61.0,14
Oakville,17320.7790,44.0,15
Vaughan,15173.8380,58.0,17
Burlington,14674.2445,33.0,10
Hamilton,12113.2105,40.0,14
Mississauga,11948.5540,27.0,8
Markham,11287.8680,36.0,14


## Step 11 – Serialization Checkpoint

I save the enriched dataset in both CSV and JSON formats. This demonstrates that the processed data can be serialized and reused outside the notebook.

In [26]:
enriched_df.to_csv(
    "../output/enriched_ecommerce.csv",
    index=False
)

enriched_df.to_json(
    "../output/enriched_ecommerce.json",
    orient="records",
    indent=4
)

print("CSV and JSON files created successfully.")

CSV and JSON files created successfully.


C:\Users\aarya\AppData\Local\Temp\ipykernel_22564\408174640.py:6: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  enriched_df.to_json(


In [27]:
json_check = pd.read_json(
    "../output/enriched_ecommerce.json"
)

json_check.head()

,date,customer_id,product,price,quantity,coupon_code,shipping_city,discount_percent,discounted_price,revenue,days_since_purchase,gross_value,city,province,country,province_code,timezone,population,latitude,longitude
0,2025-01-01,CUST1078,Tablet,583.56,3,SAVE15,Oakville,15,496.026,1488.078,345,1750.68,Oakville,Ontario,Canada,ON,America/Toronto,213759,43.4675,-79.6877
1,2025-01-01,CUST1055,Smartphone,1103.70,4,WELCOME10,Brampton,10,993.330,3973.320,345,4414.80,Brampton,Ontario,Canada,ON,America/Toronto,656480,43.7315,-79.7624
2,2025-02-01,CUST1056,Tablet,378.12,5,WELCOME10,Toronto,10,340.308,1701.540,314,1890.60,Toronto,Ontario,Canada,ON,America/Toronto,2794356,43.6532,-79.3832
3,2025-04-01,CUST1041,USB Drive,29.47,4,TECH20,Markham,20,23.576,94.304,255,117.88,Markham,Ontario,Canada,ON,America/Toronto,338503,43.8561,-79.3370
4,2025-05-01,CUST1028,Smartwatch,214.30,4,SAVE15,Windsor,15,182.155,728.620,225,857.20,Windsor,Ontario,Canada,ON,America/Toronto,233763,42.3149,-83.0364


## Step 12 – Soft Interview Reflection

Functions and classes helped organize the data-processing workflow and made the code reusable. The `Transaction` class grouped transaction-related operations, while the `DataCleaner` class kept the cleaning rules at one place. This made the notebook easier to understand and maintain. If the dataset changes, the same methods can be reused with the minimal changes.

## Data Dictionary

The Data Dictionary combines information from the primary transaction dataset and the secondary city metadata source. The datasets were connected using the shipping city. It also includes the new fields created during transformation and feature engineering.

### Final Analytical Insight

The city-level aggregation shows differences in revenue across the shipping cities. The enriched city metadata provides additional geographic information that can be used to understand the transaction data. The new discount and purchase-recency features also provide additional information for future customer and sales analysis.